In [ ]:
import pandas as pd
import numpy as np
import io

print("Step 1: Reading and Cleaning the Raw Text File...")

# 1. File ko path halne (timro Kaggle ma vako path)
file_path = '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/DGH DATA/DGH_586.txt' # Yaha aafno file ko exact naam hala

# 2. File lai line-by-line padhne
with open(file_path, 'r') as file:
    lines = file.readlines()

# 3. SMART FILTER: Jun line ma kamma (',') cha, tyo matra asali data ho!
# Baki sabai mathi ko kachara (text, years) automatic ignore huncha.
clean_lines = [line for line in lines if ',' in line]

# 4. Tyo safa lines lai eutai text block ma jodne
clean_text = "".join(clean_lines)

# 5. Pandas lai sidhai CSV jasari padhna lagaune (io.StringIO le text lai file jasto banaucha)
# Hamro data ma header chaina, tesaile aafai naam dine: 'dateTime' ra 'value'
df = pd.read_csv(io.StringIO(clean_text), header=None, names=['dateTime', 'value'])

# 6. Date format milaune (01/Jan/2006 lai Python le bujhne datetime banaune)
df['dateTime'] = pd.to_datetime(df['dateTime'], format='%d/%b/%Y')

# 7. DateTime lai index banaune (Time-series ko lagi jaruri)
df.set_index('dateTime', inplace=True)

print("✅ Data Loaded Successfully! Kachara text hatiskayo.")
print("\nFirst 5 rows of clean DataFrame:")
print(df.head())
print(f"\nTotal Data Points: {len(df)}")

In [ ]:
df.describe()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("--- OUTLIER INVESTIGATION ---")

# 1. Boxplot (Visualizing the extremes)
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['value'], color='orange')
plt.title('Distribution of Water Levels (Finding Extemes)')
plt.xlabel('Water Level (Meters)')
plt.show()

# 2. IQR Method to find mathematical outliers
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nMathematical Normal Range: {lower_bound:.2f}m to {upper_bound:.2f}m")

# 3. Let's look at the SUSPICIOUS LOWS (Top 10 lowest values)
print("\n--- TOP 10 LOWEST VALUES (Potential Sensor Drops) ---")
print(df.nsmallest(10, 'value')[['value']])

# 4. Let's look at the EXTREME HIGHS (Floods vs Errors)
print("\n--- TOP 10 HIGHEST VALUES (Potential Floods or Glitches) ---")
print(df.nlargest(10, 'value')[['value']])

In [ ]:
import pandas as pd
import numpy as np

print("--- STEP 1: REMOVING THE ZERO GLITCH ---")
# Tyo 0.00 meter (ya tyo bhanda tala ko kunai negative glitch) lai NaN banaune
df.loc[df['value'] <= 0.0, 'value'] = np.nan
print("✅ Sensor zero glitches replaced with NaN.")

print("\n--- STEP 2: FORCING A UNIFORM TIMELINE ---")
# 1. Hamro data ko suru ko din ra last ko din nikalne
start_date = df.index.min()
end_date = df.index.max()
print(f"Data Timeline: From {start_date.date()} to {end_date.date()}")

# 2. Euta perfect, continuous calendar banaune (1 day frequency)
full_timeline = pd.date_range(start=start_date, end=end_date, freq='D')

# 3. Hamro data lai yo perfect calendar sanga align (reindex) garne
# Jun din DHM le data deko chaina, tyaha Pandas le aafai NaN haldincha
df_uniform = df.reindex(full_timeline)
df_uniform.index.name = 'dateTime'

print("\n--- TIMELINE REPORT ---")
print(f"Original Data Points       : {len(df)}")
print(f"Uniform Timeline Points    : {len(df_uniform)}")
# Jati difference cha, teti ota 'hidden gaps' data ma thiyo
missing_days = df_uniform['value'].isna().sum()
print(f"Total Missing Days (NaNs)  : {missing_days}")

print("\n✅ Perfect Uniform Timeline Ready!")

In [ ]:
import numpy as np

print("--- STEP 3: SMART INTERPOLATION ---")
# 7 din samma ko gap (NaNs) lai linear interpolation le fill garne. 
# 7 din bhanda badi gap cha bhane teslai khali (NaN) chhod-dine.
gap_limit_days = 7
df_uniform['value'] = df_uniform['value'].interpolate(method='linear', limit=gap_limit_days)
print(f"Interpolation completed. Short gaps (<{gap_limit_days} days) filled.")


print("\n--- STEP 4: CHUNKING ALGORITHM (Breaking at Long Gaps) ---")
# Aba remaining NaNs (jun 7 din vanda lamo thiyo) ma data lai break garchau
def get_clean_chunks(df, min_len):
    chunks = []
    current_chunk = []
    for val in df['value'].values:
        if pd.notna(val):
            current_chunk.append(val)
        else:
            if len(current_chunk) > min_len:
                chunks.append(np.array(current_chunk))
            current_chunk = []
    if len(current_chunk) > min_len:
        chunks.append(np.array(current_chunk))
    return chunks

# Hamro daily lag time 10 days hune vako le, 10 din vanda sano chunk discard garchau
time_step = 10 
chunks = get_clean_chunks(df_uniform, min_len=time_step)

print(f"Data successfully split into {len(chunks)} continuous, safe chunks.")
for i in range(min(3, len(chunks))):
    print(f"  -> Chunk {i+1}: {len(chunks[i])} continuous days")

In [ ]:
import numpy as np

print("--- STEP 5.1: CHRONOLOGICAL TRAIN/TEST SPLIT (80/20) ---")

# Function to safely cut chunks at the 80% timeline mark
def split_chunks_chronologically(chunks_list, split_ratio=0.85):
    total_days = sum(len(c) for c in chunks_list)
    split_point = int(total_days * split_ratio)
    
    train_chunks = []
    test_chunks = []
    current_days = 0
    
    for chunk in chunks_list:
        chunk_len = len(chunk)
        
        # Condition 1: Pura chunk train ma jancha
        if current_days + chunk_len <= split_point:
            train_chunks.append(chunk)
            current_days += chunk_len
            
        # Condition 2: Chunk lai thakkai bich bata katnu parcha
        elif current_days < split_point:
            cut_idx = split_point - current_days
            train_chunks.append(chunk[:cut_idx])
            test_chunks.append(chunk[cut_idx:])
            current_days += chunk_len
            
        # Condition 3: Baki sabai chunks test (future) ma jancha
        else:
            test_chunks.append(chunk)
            
    return train_chunks, test_chunks

# Applying the split to Bagmati chunks
bagmati_train_chunks, bagmati_test_chunks = split_chunks_chronologically(chunks, split_ratio=0.8)

print(f"Total Original Days : {sum(len(c) for c in chunks)}")
print(f"Train Days (80%)    : {sum(len(c) for c in bagmati_train_chunks)}")
print(f"Test Days (20%)     : {sum(len(c) for c in bagmati_test_chunks)}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

print("\n--- STEP 5.2: LEAK-PROOF NORMALIZATION ---")

# 1. Fit Scaler strictly on TRAIN data only (Blinding it from the future)
train_combined = np.concatenate(bagmati_train_chunks).reshape(-1, 1)
bagmati_scaler = MinMaxScaler(feature_range=(0, 1))
bagmati_scaler.fit(train_combined)

# 2. Transform BOTH Train and Test chunks using the Train-fitted scaler
scaled_train_chunks = [bagmati_scaler.transform(c.reshape(-1, 1)) for c in bagmati_train_chunks]
scaled_test_chunks = [bagmati_scaler.transform(c.reshape(-1, 1)) for c in bagmati_test_chunks]

print("✅ Bagmati Normalization Complete!")
print(f"Scaler Min: {bagmati_scaler.data_min_[0]:.2f}, Max: {bagmati_scaler.data_max_[0]:.2f}")

In [ ]:
print("\n--- STEP 6: SEQUENCE EXTRACTION & VAULT STORAGE ---")

time_step = 10 

def extract_windows(scaled_chunks_list, time_step):
    X_list, y_list = [], []
    for chunk in scaled_chunks_list:
        # Check if chunk is large enough to create at least one window
        if len(chunk) > time_step:
            for i in range(len(chunk) - time_step):
                X_list.append(chunk[i:(i + time_step), 0])
                y_list.append(chunk[i + time_step, 0])
    return np.array(X_list), np.array(y_list)

# Generate Train Sequences
X_train_bagmati, y_train_bagmati = extract_windows(scaled_train_chunks, time_step)
X_train_bagmati = X_train_bagmati.reshape(X_train_bagmati.shape[0], X_train_bagmati.shape[1], 1)

# Generate Test Sequences
X_test_bagmati, y_test_bagmati = extract_windows(scaled_test_chunks, time_step)
X_test_bagmati = X_test_bagmati.reshape(X_test_bagmati.shape[0], X_test_bagmati.shape[1], 1)

# Store everything in a master dictionary for Bagmati
bagmati_vault = {
    'train': (X_train_bagmati, y_train_bagmati),
    'test': (X_test_bagmati, y_test_bagmati),
    'scaler': bagmati_scaler
}

print("✅ Bagmati Vault Successfully Created!")
print("-" * 50)
print(f"Train Shape: X={X_train_bagmati.shape}, y={y_train_bagmati.shape}")
print(f"Test Shape : X={X_test_bagmati.shape}, y={y_test_bagmati.shape}")
print("-" * 50)

# Optional: Clean up memory
del chunks
del bagmati_train_chunks
del bagmati_test_chunks
del scaled_train_chunks
del scaled_test_chunks

In [ ]:
import pandas as pd
import io

print("--- STEP 1: SMART PARALLEL LOADING (TXT & CSV Supported) ---")

# Timro station IDs ra exact file names (Kaggle ko path anusar update gara)
sunkoshi_stations = {
    '630': '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/DGH DATA/DGH_630.txt',   # TXT format
    '652': '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/DGH DATA/DGH_652.txt',   # TXT format
    '665': '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/665 Water Level Inst from 2019-01-01 to 2026-06-15.csv'    # CSV format
}

# Yo naya dictionary le Sunkoshi ko data lai Bagmati vanda 100% separate rakhcha
sunkoshi_raw_data = {}

for station_id, filepath in sunkoshi_stations.items():
    print(f"\nReading Station {station_id} from {filepath}...")
    try:
        # FORMAT 1: Handle TXT files with metadata headers
        if filepath.lower().endswith('.txt'):
            with open(filepath, 'r') as file:
                clean_lines = [line for line in file.readlines() if ',' in line]
            clean_text = "".join(clean_lines)
            df = pd.read_csv(io.StringIO(clean_text), header=None, names=['dateTime', 'value'])
            # TXT ko date format '01/Jan/2006' jasto huncha
            df['dateTime'] = pd.to_datetime(df['dateTime'], format='%d/%b/%Y', errors='coerce')

        # FORMAT 2: Handle standard CSV files
        elif filepath.lower().endswith('.csv'):
            df = pd.read_csv(filepath)
            # CSV ma header farak huna sakcha (e.g., 'Date', 'WaterLevel'). 
            # Safe huna ko lagi suru ko 2 wata column lai 'dateTime' ra 'value' manam
            df = df.iloc[:, [0, 1]] 
            df.columns = ['dateTime', 'value']
            # Pandas lai aafai date format guess garna dine
            df['dateTime'] = pd.to_datetime(df['dateTime'], errors='coerce')
        
        else:
            print(f"⚠️ Warning: {filepath} ko format milena. Skipping...")
            continue

        # COMMON CLEANUP FOR BOTH FORMATS
        # 1. Khali dates ya values falne
        df.dropna(subset=['dateTime', 'value'], inplace=True)
        # 2. Value lai strictly Number (float) banaune taaki text naveti-os
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df.dropna(subset=['value'], inplace=True)
        # 3. Index set garne ra date anusar sort garne
        df.set_index('dateTime', inplace=True)
        df.sort_index(inplace=True)
        
        # Finally, safely dictionary ma save garne
        sunkoshi_raw_data[station_id] = df
        
        print(f"✅ Station {station_id} Locked & Loaded: {len(df)} records found.")
        
    except FileNotFoundError:
        print(f"❌ Error: {filepath} not found in Kaggle! Check the name again.")
    except Exception as e:
        print(f"❌ Error processing Station {station_id}: {e}")

print("\n-----------------------------------------------------------")
print("✅ All Sunkoshi data safely isolated in 'sunkoshi_raw_data'.")
print("Bagmati data remains completely untouched and safe.")
print("-----------------------------------------------------------")

In [ ]:
sunkoshi_raw_data["665"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- DEEP DIVE: ANALYZING 'OUTLIERS' BEFORE DELETING ---")

def inspect_extreme_peaks(df, station_id, threshold):
    print(f"\n🔍 Station {station_id}: Values above {threshold}m")
    
    # 1. Paila daily resample garau (taaki raw 10-min noise hato)
    df_daily = df[['value']].resample('D').mean()
    
    # 2. Extract values above threshold
    high_values = df_daily[df_daily['value'] > threshold]
    
    # if high_values.empty:
    #     print(f"No values above threshold.")
    #     return

    # 3. Print the top 5 highest to see if they are isolated days or continuous
    top_5 = high_values.nlargest(5, 'value')
    print("Top 5 Highest Records (Check Dates for continuity):")
    print(top_5)

    # 4. Plot the whole series with the threshold line to see context
    plt.figure(figsize=(15, 4))
    plt.plot(df_daily.index, df_daily['value'], color='blue', label='Daily Water Level')
    plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold}m)')
    
    # Highlight the extreme peaks in orange
    plt.scatter(high_values.index, high_values['value'], color='orange', s=30, label='Extreme Peaks', zorder=5)

    plt.title(f'Sunkoshi River (Station {station_id}) - True Flood vs Glitch Analysis')
    plt.ylabel('Water Level (m)')
    plt.legend()
    plt.grid(True)
    plt.show()

# Run the inspection for the most problematic stations
inspect_extreme_peaks(sunkoshi_raw_data['652'], '652', threshold=7.0)
inspect_extreme_peaks(sunkoshi_raw_data['665'], '665', threshold=7.0)
inspect_extreme_peaks(sunkoshi_raw_data['630'], '630', threshold=7.0)


In [ ]:
import pandas as pd
import numpy as np

print("--- RE-RUNNING CLEANING WITH SMART HYDROLOGICAL LIMITS ---")

# Define our absolute safe boundaries
ABSOLUTE_MIN = 0.01  # Zero or negative is always a sensor glitch
ABSOLUTE_MAX = 12.0 # Anything above 12m in Sunkoshi is likely a sensor error

for station_id, df in sunkoshi_raw_data.items():
    # 1. Resampling to Daily Mean (Harmonization)
    df_daily = df[['value']].resample('D').mean()
    
    # 2. Applying Smart Limits
    # Identify glitches outside our absolute boundaries
    outlier_mask = (df_daily['value'] <= ABSOLUTE_MIN) | (df_daily['value'] > ABSOLUTE_MAX)
    total_outliers = outlier_mask.sum()
    
    # Replace glitches with NaN
    df_daily.loc[outlier_mask, 'value'] = np.nan
    
    # 3. Forcing Uniform Timeline
    start_date = df_daily.index.min()
    end_date = df_daily.index.max()
    full_timeline = pd.date_range(start=start_date, end=end_date, freq='D')
    df_uniform = df_daily.reindex(full_timeline)
    
    # 4. Save back to our dictionary
    sunkoshi_raw_data[station_id] = df_uniform
    
    print(f"✅ Station {station_id}: Resampled & Cleaned. Removed {total_outliers} extreme outliers.")

print("\nAll stations are now cleaned with smart limits and have a uniform timeline.")

In [ ]:
sunkoshi_raw_data["665"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- VERIFYING CLEAN DATA WITH BOXPLOTS ---")

# Eutai figure ma 3 wata plot banaune setup
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Sunkoshi River: Boxplots After Smart Cleaning (0m - 12m Limit)', fontsize=16, fontweight='bold')

# Dictionary ko keys (station IDs) nikalne
stations = list(sunkoshi_raw_data.keys())

for i, station_id in enumerate(stations):
    df_clean = sunkoshi_raw_data[station_id]
    
    # Boxplot draw garne
    sns.boxplot(y=df_clean['value'], ax=axes[i], color='lightgreen', fliersize=4)
    
    # Graph ko design milaune
    axes[i].set_title(f'Station {station_id}', fontsize=14)
    axes[i].set_ylabel('Water Level (m)', fontsize=12)
    
    # Y-axis lai thakkai -1 dekhi 13 samma lock garne taaki 0-12 ko range majjale dekhayos
    axes[i].set_ylim(-1, 13) 
    
    # Grid thapne easy reading ko lagi
    axes[i].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Print a quick text summary to confirm max/min numerically
print("\n--- FINAL NUMERICAL VERIFICATION ---")
for station_id in stations:
    df_clean = sunkoshi_raw_data[station_id]
    print(f"Station {station_id} -> Min: {df_clean['value'].min():.2f}m | Max: {df_clean['value'].max():.2f}m")

In [ ]:
import pandas as pd
import numpy as np

print("--- STEP 3: FORCING UNIFORM TIMELINE (Exposing Hidden Gaps) ---")

for station_id, df in sunkoshi_raw_data.items():
    # 1. Station ko purano stats note garne
    original_len = len(df)
    start_date = df.index.min()
    end_date = df.index.max()
    
    # 2. Perfect continuous daily calendar banaune (Start to End)
    full_timeline = pd.date_range(start=start_date, end=end_date, freq='D')
    
    # 3. Data lai tyo perfect calendar sanga align (reindex) garne
    # Jun dates paila thiyena, tyaha aafai NaN aayera bascha
    df_uniform = df.reindex(full_timeline)
    df_uniform.index.name = 'dateTime'
    
    # 4. Dictionary ma clean uniform data update garne
    sunkoshi_raw_data[station_id] = df_uniform
    
    # 5. Report Print Garne
    uniform_len = len(df_uniform)
    hidden_gaps = uniform_len - original_len
    total_nans = df_uniform['value'].isna().sum()
    
    print(f"\n==========================================")
    print(f"🗓️ STATION {station_id} TIMELINE REPORT")
    print(f"==========================================")
    print(f"Timeline Date Range : {start_date.date()} to {end_date.date()}")
    print(f"Original Days Count : {original_len}")
    print(f"Perfect Days Count  : {uniform_len}")
    print(f"Hidden Gaps Exposed : {hidden_gaps} days (Now NaN)")
    print(f"Total NaNs in data  : {total_nans} (Outliers + Gaps)")

print("\n-----------------------------------------------------------")
print("✅ Phase 3 Complete: All stations are now mapped to a perfect daily calendar!")
print("-----------------------------------------------------------")

In [ ]:
import numpy as np
import pandas as pd

print("--- STEP 4: SMART INTERPOLATION & CHUNKING ONLY ---")

gap_limit_days = 7
time_step = 10 

# Naya dictionary jasma harek station ko aafnai chunks haru safe baschan
sunkoshi_chunks_dict = {}

for station_id, df in sunkoshi_raw_data.items():
    print(f"\nProcessing Station {station_id}...")
    
    # 1. INTERPOLATION (Max 7 days)
    initial_nans = df['value'].isna().sum()
    df['value'] = df['value'].interpolate(method='linear', limit=gap_limit_days)
    remaining_nans = df['value'].isna().sum()
    print(f" -> Interpolated {initial_nans - remaining_nans} small gaps.")
    print(f" -> Remaining large NaNs (Cut points): {remaining_nans}")
    
    # 2. CHUNKING ALGORITHM (Breaking at remaining NaNs)
    station_chunks = []
    current_chunk = []
    
    for val in df['value'].values:
        if pd.notna(val):
            current_chunk.append(val)
        else:
            if len(current_chunk) > time_step:
                station_chunks.append(np.array(current_chunk))
            current_chunk = [] # Reset for next block
            
    # Check for the last remaining chunk after the loop ends
    if len(current_chunk) > time_step:
        station_chunks.append(np.array(current_chunk))
        
    # Safely store chunks in the new dictionary
    sunkoshi_chunks_dict[station_id] = station_chunks
    
    print(f" -> Chunking complete: Generated {len(station_chunks)} safe chunks.")
    # Sano preview matra dekhaune
    for i in range(min(3, len(station_chunks))):
        print(f"    - Chunk {i+1} length: {len(station_chunks[i])} days")

print("\n-----------------------------------------------------------")
print("✅ Step 4 Complete: All stations are interpolated and safely chunked!")
print("-----------------------------------------------------------")

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("--- STEP 5 & 6: LEAK-PROOF SPLIT, SCALING & SEQUENCING FOR SUNKOSHI ---")

# HELPER FUNCTIONS (Same safe logic we used for Bagmati)
def split_chunks_chronologically(chunks_list, split_ratio=0.8):
    total_days = sum(len(c) for c in chunks_list)
    split_point = int(total_days * split_ratio)
    train_chunks, test_chunks = [], []
    current_days = 0
    
    for chunk in chunks_list:
        chunk_len = len(chunk)
        if current_days + chunk_len <= split_point:
            train_chunks.append(chunk)
            current_days += chunk_len
        elif current_days < split_point:
            cut_idx = split_point - current_days
            train_chunks.append(chunk[:cut_idx])
            test_chunks.append(chunk[cut_idx:])
            current_days += chunk_len
        else:
            test_chunks.append(chunk)
    return train_chunks, test_chunks

def extract_windows(scaled_chunks_list, time_step):
    X_list, y_list = [], []
    for chunk in scaled_chunks_list:
        if len(chunk) > time_step:
            for i in range(len(chunk) - time_step):
                X_list.append(chunk[i:(i + time_step), 0])
                y_list.append(chunk[i + time_step, 0])
    return np.array(X_list), np.array(y_list)

time_step = 10
# Naya Vault jasma Sunkoshi ko sabai station ko isolated Train/Test data ra Scalers baschan
sunkoshi_vault = {} 

for station_id, station_chunks in sunkoshi_chunks_dict.items():
    print(f"\n==========================================")
    print(f"⚙️ PROCESSING STATION: {station_id}")
    
    # 1. TIME-KNIFE SPLIT (80% Train, 20% Test)
    train_chunks, test_chunks = split_chunks_chronologically(station_chunks, split_ratio=0.85)
    
    # 2. LEAK-PROOF NORMALIZATION (Independent for this station)
    train_combined = np.concatenate(train_chunks).reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(train_combined) # Fitting only on PAST data!
    
    scaled_train_chunks = [scaler.transform(c.reshape(-1, 1)) for c in train_chunks]
    scaled_test_chunks = [scaler.transform(c.reshape(-1, 1)) for c in test_chunks]
    
    # 3. SEQUENCE EXTRACTION
    X_train, y_train = extract_windows(scaled_train_chunks, time_step)
    X_test, y_test = extract_windows(scaled_test_chunks, time_step)
    
    # LSTM ko lagi 3D shape ma laijane
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    if len(X_test) > 0:
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    
    # 4. STORE IN STATION VAULT
    sunkoshi_vault[station_id] = {
        'train': (X_train, y_train),
        'test': (X_test, y_test),
        'scaler': scaler
    }
    
    print(f" -> Train Sequences : {len(X_train)}")
    print(f" -> Test Sequences  : {len(X_test)}")
    print(f" -> Scaler Fits     : Min={scaler.data_min_[0]:.2f}, Max={scaler.data_max_[0]:.2f}")

print("\n-----------------------------------------------------------")
print("✅ SUNKOSHI VAULT SECURED: All 3 stations are leak-proofed & sequenced!")
print("-----------------------------------------------------------")

# RAM free garna ko lagi purano variables falne
del sunkoshi_chunks_dict

In [ ]:
import pandas as pd
import numpy as np
import io

print("--- STEP 1: LOAD, CLEAN & UNIFORM TIMELINE (LIKHU & KOKHAJOR) ---")

# Timro text files ko exact Kaggle names halnu hai
extra_stations = {
    'likhu': '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/DGH DATA/DGH_660.txt',      
    'kokhajor': '/kaggle/input/datasets/shishirpuri/kamala-master-model/Hydrology/DGH DATA/DGH_580.txt' 
}

# Yesma hamro clean uniform data bascha
extra_rivers_raw_data = {}
ABSOLUTE_MAX = 15.0 # Khola ko lagi 15m vanda mathi 100% glitch ho

for name, filepath in extra_stations.items():
    print(f"\n🌊 Processing {name.upper()}...")
    try:
        # 1. Text bata kachara hatayera load garne
        with open(filepath, 'r') as file:
            clean_lines = [line for line in file.readlines() if ',' in line]
        df = pd.read_csv(io.StringIO("".join(clean_lines)), header=None, names=['dateTime', 'value'])
        
        # 2. Date ra numbers milaune
        df['dateTime'] = pd.to_datetime(df['dateTime'], format='%d/%b/%Y', errors='coerce')
        df.dropna(subset=['dateTime', 'value'], inplace=True)
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df.set_index('dateTime', inplace=True)
        df.sort_index(inplace=True)
        
        # 3. Daily Mean (Harmonization) ra Glitch Removal (<=0 ya >15m)
        df_daily = df[['value']].resample('D').mean()
        outliers = (df_daily['value'] <= 0.0) | (df_daily['value'] > ABSOLUTE_MAX)
        df_daily.loc[outliers, 'value'] = np.nan
        
        # 4. Forcing Perfect Uniform Calendar
        start_date = df_daily.index.min()
        end_date = df_daily.index.max()
        full_timeline = pd.date_range(start=start_date, end=end_date, freq='D')
        df_uniform = df_daily.reindex(full_timeline)
        df_uniform.index.name = 'dateTime'
        
        # Safe dict ma save garne
        extra_rivers_raw_data[name] = df_uniform
        
        print(f" -> Timeline: {start_date.date()} to {end_date.date()}")
        print(f" -> Total Days (Uniform Timeline): {len(df_uniform)}")
        print(f" -> Total Missing/NaNs (Gaps + Outliers): {df_uniform['value'].isna().sum()}")
        
    except FileNotFoundError:
        print(f" -> ❌ Error: {filepath} not found in Kaggle! Check the name.")
    except Exception as e:
        print(f" -> ❌ Error processing {name}: {e}")

print("\n-----------------------------------------------------------")
print("✅ Step 1 Complete: Likhu and Kokhajor are loaded and mapped to calendar!")
print("-----------------------------------------------------------")

In [ ]:
 extra_rivers_raw_data['kokhajor'].describe()

In [ ]:
import numpy as np
import pandas as pd

print("--- STEP 2: SMART INTERPOLATION & CHUNKING (LIKHU & KOKHAJOR) ---")

gap_limit_days = 7
time_step = 10 

# Naya dictionary jasma Likhu ra Kokhajor ko safe chunks baschan
extra_rivers_chunks_dict = {}

for station_name, df in extra_rivers_raw_data.items():
    print(f"\nProcessing {station_name.upper()}...")
    
    # 1. INTERPOLATION (Max 7 days gaps fill garne)
    initial_nans = df['value'].isna().sum()
    df['value'] = df['value'].interpolate(method='linear', limit=gap_limit_days)
    remaining_nans = df['value'].isna().sum()
    print(f" -> Interpolated {initial_nans - remaining_nans} small gaps/outliers.")
    print(f" -> Remaining large NaNs (Cut points): {remaining_nans}")
    
    # 2. CHUNKING ALGORITHM (Breaking data at large NaNs)
    station_chunks = []
    current_chunk = []
    
    for val in df['value'].values:
        if pd.notna(val):
            current_chunk.append(val)
        else:
            if len(current_chunk) > time_step:
                station_chunks.append(np.array(current_chunk))
            current_chunk = [] # Reset for next block
            
    # Loop sakiye pachi last ko chunk check garne
    if len(current_chunk) > time_step:
        station_chunks.append(np.array(current_chunk))
        
    # Dictionary ma chunks save garne
    extra_rivers_chunks_dict[station_name] = station_chunks
    
    print(f" -> Chunking complete: Generated {len(station_chunks)} safe chunks.")
    for i in range(min(3, len(station_chunks))):
        print(f"    - Chunk {i+1} length: {len(station_chunks[i])} days")

print("\n-----------------------------------------------------------")
print("✅ Step 2 Complete: Likhu and Kokhajor are safely chunked!")
print("-----------------------------------------------------------")

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("--- STEP 3: LEAK-PROOF SPLIT, SCALING & SEQUENCES ---")

# HELPER FUNCTIONS
def split_chunks_chronologically(chunks_list, split_ratio=0.8):
    total_days = sum(len(c) for c in chunks_list)
    split_point = int(total_days * split_ratio)
    train_chunks, test_chunks = [], []
    current_days = 0
    
    for chunk in chunks_list:
        chunk_len = len(chunk)
        if current_days + chunk_len <= split_point:
            train_chunks.append(chunk)
            current_days += chunk_len
        elif current_days < split_point:
            cut_idx = split_point - current_days
            train_chunks.append(chunk[:cut_idx])
            test_chunks.append(chunk[cut_idx:])
            current_days += chunk_len
        else:
            test_chunks.append(chunk)
    return train_chunks, test_chunks

def extract_windows(scaled_chunks_list, time_step):
    X_list, y_list = [], []
    for chunk in scaled_chunks_list:
        if len(chunk) > time_step:
            for i in range(len(chunk) - time_step):
                X_list.append(chunk[i:(i + time_step), 0])
                y_list.append(chunk[i + time_step, 0])
    return np.array(X_list), np.array(y_list)

time_step = 10
# Naya Vault jasma Likhu ra Kokhajor ko Train/Test sequences ra Scalers baschan
extra_rivers_vault = {} 

for station_id, station_chunks in extra_rivers_chunks_dict.items():
    print(f"\n==========================================")
    print(f"⚙️ PROCESSING {station_id.upper()}")
    
    # 1. TIME-KNIFE SPLIT (80% Train, 20% Test)
    train_chunks, test_chunks = split_chunks_chronologically(station_chunks, split_ratio=0.8)
    
    # 2. LEAK-PROOF NORMALIZATION (Independent for this station)
    train_combined = np.concatenate(train_chunks).reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(train_combined) # Fits ONLY on Train data
    
    scaled_train_chunks = [scaler.transform(c.reshape(-1, 1)) for c in train_chunks]
    scaled_test_chunks = [scaler.transform(c.reshape(-1, 1)) for c in test_chunks]
    
    # 3. SEQUENCE EXTRACTION (10-Day Lag)
    X_train, y_train = extract_windows(scaled_train_chunks, time_step)
    X_test, y_test = extract_windows(scaled_test_chunks, time_step)
    
    # Reshape for LSTM (3D format)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    if len(X_test) > 0:
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    
    # 4. STORE IN EXTRA RIVERS VAULT
    extra_rivers_vault[station_id] = {
        'train': (X_train, y_train),
        'test': (X_test, y_test),
        'scaler': scaler
    }
    
    print(f" -> Train Sequences : {len(X_train)}")
    print(f" -> Test Sequences  : {len(X_test)}")
    print(f" -> Scaler Fits     : Min={scaler.data_min_[0]:.2f}, Max={scaler.data_max_[0]:.2f}")

print("\n-----------------------------------------------------------")
print("✅ EXTRA RIVERS VAULT SECURED: Likhu and Kokhajor are leak-proofed & sequenced!")
print("-----------------------------------------------------------")

# Free up memory (Memory Management is crucial in ML)
del extra_rivers_raw_data
del extra_rivers_chunks_dict

In [ ]:
import numpy as np
from sklearn.utils import shuffle

print("--- STEP 4: THE GRAND MASTER MERGE ---")

# Global variables for training
X_train_list = []
y_train_list = []

# Naya master vault jasma sabai station ko Test Data ra Scaler matra bascha
master_evaluation_vault = {}

# Helper function to extract train data and safely store test data
def merge_and_store(station_name, vault_data):
    # 1. Train data lai Global list ma thapne
    X_tr, y_tr = vault_data['train']
    X_train_list.append(X_tr)
    y_train_list.append(y_tr)
    
    # 2. Test data ra Scaler lai Master Vault ma safe rakhne
    X_te, y_te = vault_data['test']
    master_evaluation_vault[station_name] = {
        'X_test': X_te,
        'y_test': y_te,
        'scaler': vault_data['scaler']
    }
    print(f" -> {station_name.upper()} added. (Train: {len(X_tr)}, Test: {len(X_te)})")

print("\n📦 Packing Bagmati...")
merge_and_store('bagmati', bagmati_vault)

print("\n📦 Packing Sunkoshi Stations...")
for station_id, data in sunkoshi_vault.items():
    merge_and_store(f'sunkoshi_{station_id}', data)

print("\n📦 Packing Extra Rivers (Likhu & Kokhajor)...")
for station_id, data in extra_rivers_vault.items():
    merge_and_store(station_id, data)

# ==========================================
# FINAL CONCATENATION & SHUFFLING
# ==========================================
print("\n🔄 Merging and Shuffling Global Train Data...")

# Sabai nadi ko array lai eutai thulo array ma jodne
X_train_global = np.concatenate(X_train_list, axis=0)
y_train_global = np.concatenate(y_train_list, axis=0)

# Model lai kunai nadi ko specific order yaad nahos bhanera majjale shuffle garne
X_train_global, y_train_global = shuffle(X_train_global, y_train_global, random_state=42)

print("\n=======================================================")
print("✅ UNIVERSAL BASE DATASET READY FOR NEURAL NETWORK!")
print(f"Shape of X_train_global : {X_train_global.shape}")
print(f"Shape of y_train_global : {y_train_global.shape}")
print(f"Total Stations in Vault : {len(master_evaluation_vault.keys())}")
print("=======================================================")

# Free up memory (Discarding the old intermediate vaults)
del bagmati_vault
del sunkoshi_vault
del extra_rivers_vault
del X_train_list
del y_train_list

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("--- BUILDING THE UNIVERSAL BASE MODEL ---")

# 1. Define Model Architecture
model = Sequential()

# Layer 1: First LSTM Layer 
# return_sequences=True is crucial here because we are passing the sequence to another LSTM layer
model.add(LSTM(units=64, return_sequences=True, input_shape=(X_train_global.shape[1], 1)))
model.add(Dropout(0.2)) # Randomly drops 20% connections to prevent memorization (overfitting)

# Layer 2: Second LSTM Layer
# return_sequences=False because we now just want the final extracted pattern, not the whole sequence
model.add(LSTM(units=32, return_sequences=False))
model.add(Dropout(0.2))

# Layer 3: Fully Connected Dense Layer
model.add(Dense(units=16, activation='relu'))

# Output Layer: 1 Neuron for predicting the scaled water level (0 to 1)
model.add(Dense(units=1, activation='linear'))

# 2. Compile the Model
# Adam optimizer adaptively changes learning rate. 
# MSE (Mean Squared Error) is the standard loss for continuous values.
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='mse', 
              metrics=['mae']) # MAE (Mean Absolute Error) for easier human interpretation

print("✅ Model Architecture Ready!\n")
model.summary()

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("--- STEP 6: TRAINING THE UNIVERSAL MODEL ---")

# 1. Setup Smart Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,               # 10 epochs samma improvement navaye rokidina
    restore_best_weights=True, # Sabai bhanda best epoch ko dimag save garne
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,                # LR lai 50% le ghataune
    patience=5,                # 5 epochs samma improve navaye LR ghataune
    min_lr=0.00001, 
    verbose=1
)

EPOCHS = 100
BATCH_SIZE = 64

print("🚀 Starting Training... Wait for a few minutes!\n")

# 2. Train the Model
history = model.fit(
    X_train_global, y_train_global,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2, # Uses 20% of the shuffled training pool for internal validation
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n✅ Training Complete! The model has learned the Universal River Patterns.")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='blue', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='orange', linewidth=2)

plt.title('Base Model Learning Curve: Training vs Validation Loss', fontsize=16, fontweight='bold')
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Mean Squared Error (Scaled 0-1)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

print("--- UNLOCKING THE VAULT: REAL WORLD EVALUATION ---")

# Hami just first 3 stations matra plot garchau to avoid clutter
stations_to_plot = list(master_evaluation_vault.keys())[:3]

for station in stations_to_plot:
    data = master_evaluation_vault[station]
    X_test = data['X_test']
    y_test_scaled = data['y_test']
    scaler = data['scaler']
    
    if len(X_test) == 0:
        print(f"Skipping {station} (No test sequences)")
        continue
        
    print(f"\n🔮 Predicting Future for {station.upper()}...")
    
    # 1. Model makes predictions on the unseen scaled data
    predictions_scaled = model.predict(X_test, verbose=0)
    
    # 2. INVERSE TRANSFORM (Back to Real Meters)
    predictions_real = scaler.inverse_transform(predictions_scaled)
    y_test_real = scaler.inverse_transform(y_test_scaled.reshape(-1, 1))
    
    # 3. Plotting the Actual vs Predicted Hydrograph
    plt.figure(figsize=(14, 5))
    
    # Hami clarity ko lagi test data ko specific slice (e.g., 200 days) matra plot garchau
    plot_range = min(200, len(y_test_real)) 
    
    plt.plot(y_test_real[:plot_range], label='Actual Water Level (m)', color='blue', linewidth=2)
    plt.plot(predictions_real[:plot_range], label='Predicted Water Level (m)', color='red', linestyle='--', linewidth=2)
    
    plt.title(f'{station.upper()} - Next Day Water Level Prediction (First {plot_range} Days of Test Data)', fontsize=14, fontweight='bold')
    plt.xlabel('Days into the Future (Test Set)', fontsize=12)
    plt.ylabel('Water Level (Meters)', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

print("\n✅ Real-World Evaluation Complete!")